## Generators & Augmentation testing

In this notebook we will try new augmentation algorithm over the cells

In [ ]:
import pandas as pd
from composition.cells import CELLS

hungry = set(pd.read_parquet("data/composition/cell_order_sheet.parquet")["floor"])
audit = pd.DataFrame([
    {"cell": c.name, "hungry": c.name in hungry, "looks_like": c.looks_like}
    for c in CELLS
]).assign(written=lambda d: d["looks_like"].notna())

print(f"{audit['written'].sum()} of {len(audit)} cells have looks_like")
print(f"hungry cells still missing one: {(audit['hungry'] & ~audit['written']).sum()}")

audit[~audit["written"]].sort_values("hungry", ascending=False)[["cell", "hungry"]]

In [ ]:
audit[audit["written"]]

In [ ]:
import pandas as pd

from augmentation.config import AugmentationPaths
from augmentation.loop import AugmentationLoop
from augmentation.parents import ParentPool
from dataset_registry import DATASETS


In [ ]:
paths = AugmentationPaths()
catalog = pd.read_parquet(paths.catalog).astype({"query_id": str})
selection = pd.read_parquet(paths.data_dir / "composition/cell_selection.parquet").astype({"query_id": str})
parents = ParentPool(catalog, selection, {d.name: d for d in DATASETS})

In [ ]:
loop = AugmentationLoop(selection, sheet_path=paths.cell_order_sheet, parents=parents)
loop.plans(parents.available())

In [ ]:
CELL = "version_pinned_technical"          # two mints — exercises the pipeline
result, frame = loop.demand(CELL)

# grounded() attaches the gold document, but only when the plan CUTS — the cut
# has to keep that document answering, so it reads it (d53d)
parent = loop.grounded(result, frame.iloc[0])
parent[["dataset", "query_id", "query", "surfaces", "bank"]]

In [ ]:
# d55: there is no longer ONE instruction per row. A cell becomes a sequence of
# calls — every addition in the first, the cut in a second — and each call's
# targets accumulate, so a later call cannot undo an earlier one.
from augmentation.dispatch import calls_for, targets_of
from composition.cells import CELLS_BY_NAME

for i, call in enumerate(calls_for(result, CELLS_BY_NAME[CELL], parent), 1):
    mints = [s.operator.declaration.operator for s in call.steps]
    print(f"{'=' * 25} CALL {i}  mints={mints}  cuts={call.cuts} {'=' * 25}")
    print(loop.brief(CELL, call.steps, parent))
    print("\nVERIFIED:", targets_of(call.verified, parent).model_dump_json(indent=1))
    print()

In [ ]:
from augmentation.pool import GeneratedPool
from augmentation.qrels import AugmentationQrels

In [ ]:
throwaway = AugmentationPaths(data_dir="/tmp/smoke")

loop = AugmentationLoop(
    selection, sheet_path=paths.cell_order_sheet, parents=parents,
    pool=GeneratedPool(throwaway), qrels=AugmentationQrels(throwaway),
)

In [ ]:

loop.run("version_pinned_technical", n=1)

In [ ]:
loop.run("symbol_pile_no_grammar", n=1)

In [ ]:
loop.run("legal_citation_canonical", n=5)

In [ ]:
from augmentation.campaign import AugmentationCampaign

campaign = AugmentationCampaign(loop)
campaign.plan()

In [ ]:
import contextlib
from pathlib import Path

log_path = Path("/tmp/campaign_run.log")
with log_path.open("w") as f, contextlib.redirect_stdout(f):
    summary = campaign.run()

print(f"done — {len(summary)} floors, full log at {log_path}")

In [ ]:
import pandas as pd
pool = pd.read_parquet("data/augmentation/pool.parquet")
pool["floor"].value_counts()   # rows per cell
pool[["query_id", "query", "floor", "credit_gate"]].tail(20)